# 01 - What the data looks like

**Scope discipline.** Everything that looks for *patterns* here uses only the 110
training timestamps. The 24 holdout observations are never inspected in this
notebook: that is the point of `docs/protocol.md`. Structural checks (schema,
grid, duplicates) are allowed over the whole file because they do not reveal
target values.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 120)
ART = ROOT / "reports" / "artifacts"
print("project root:", ROOT)

project root: C:\Z_Coding\Cloud-Workload-Forecasting


## Structural facts

Read the readings file and check the grid, duplicates and value ranges. These are
integrity questions, not forecasting questions, so the full file is fair game.

In [2]:
readings = pd.read_csv(ROOT / "challenge_data" / "vm_cpu_readings.csv.gz")
print("rows:", len(readings))
print("vms :", readings.vm_id.nunique())
ts = np.sort(readings.timestamp.unique())
print("timestamps:", len(ts))
print("gaps (s):", np.unique(np.diff(ts)))
print("duplicate (vm,timestamp) keys:", readings.duplicated(["vm_id", "timestamp"]).sum())
print("avg_cpu outside [0,100]:", (~readings.avg_cpu.between(0, 100)).sum())

rows: 1340000
vms : 10000
timestamps: 134
gaps (s): [300]


duplicate (vm,timestamp) keys: 0
avg_cpu outside [0,100]: 0


## The training window only

`cwf.splits.chronological_split` is the single owner of the split. From here on
we look at `split.train` and never at `split.holdout`.

In [3]:
from cwf.data import load_dataset
from cwf.targets import build_targets
from cwf.splits import chronological_split

series = build_targets(load_dataset(ROOT / "challenge_data").readings)
split = chronological_split(series, holdout_len=24, train_len=None)
train = split.train
print("train:", len(train), "holdout:", len(split.holdout))
print("holdout timestamps are known, their values are not touched")

train[["timestamp", "active_vm_count", "aggregate_cpu_utilization"]].describe().T

train: 110 holdout: 24
holdout timestamps are known, their values are not touched


,count,mean,std,min,25%,50%,75%,max
timestamp,110.0,16350.000000,9569.482745,0.000000,8175.000000,16350.000000,24525.000000,32700.000000
active_vm_count,110.0,10000.000000,0.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
aggregate_cpu_utilization,110.0,9.414089,0.193568,9.111634,9.288952,9.378121,9.507673,9.912175


**Read this carefully.** `active_vm_count` has std 0: it is exactly 10,000 at all
110 training timestamps. That is a property of a complete rectangular release
slice, not of Azure. It means the count target is trivially persistent, and it
means every scale-free metric for that target divides by zero. We report that
honestly rather than manufacture complexity.

In [4]:
print("distinct active counts in training:", train.active_vm_count.nunique())
print("aggregate CPU summary:")
print(train.aggregate_cpu_utilization.describe())

distinct active counts in training: 1
aggregate CPU summary:
count    110.000000
mean       9.414089
std        0.193568
min        9.111634
25%        9.288952
50%        9.378121
75%        9.507673
max        9.912175
Name: aggregate_cpu_utilization, dtype: float64


## Is there trend or seasonality?

Two unit-root tests plus an autocorrelation function. With 9.2 hours we cannot
see a daily cycle, so we are asking a narrower question: is there a drift we can
exploit, and how fast does information decay?

In [5]:
from cwf.data import load_dataset as _ld
import numpy as _np

acf = pd.read_csv(ART / "acf_train.csv")
print(acf.head(26).to_string(index=False))

 lag       acf  lag_minutes
   1  0.662068          5.0
   2  0.594555         10.0
   3  0.663980         15.0
   6  0.449758         30.0
  12  0.327413         60.0
  24 -0.019925        120.0
  36 -0.186841        180.0
  48 -0.019542        240.0
  72  0.102255        360.0


In [6]:
print(pd.read_csv(ART / "stationarity_trend.csv").to_string(index=False))
print()
print(pd.read_csv(ART / "acf_decay.csv").to_string(index=False))

 trend_slope_pp_per_hour  trend_slope_se_pp_per_hour  trend_t_stat  adf_pvalue  kpss_pvalue  adf_pvalue_diff  kpss_pvalue_diff  acf_lag1  acf_lag12  acf_lag24  ci_95_acf_band  n_training
               -0.012691                     0.00966     -1.313743    0.413266          0.1          0.07742               0.1  0.662068   0.327413  -0.019925        0.186879         110

 first_lag_inside_95pct_band  minutes_to_decay
                          11              55.0


Autocorrelation is still 0.66 at lag 1 but drops inside the 95% band by lag 11
(about 55 minutes). ADF says "unit root not rejected", KPSS says "stationary not
rejected" at the 10% level: the series is close to the boundary, i.e. very
persistent but not obviously trending. Practically: last value and short averages
are both plausible, and neither will be much better than the other.

## The mean hides the tail

The aggregate is about 9.4 percentage points. Individual machines reach 99.
Capacity planning on the mean alone would be a mistake.

In [7]:
print(pd.read_csv(ART / "mean_hides_tails.csv").to_string(index=False))
print()
print(pd.read_csv(ART / "load_concentration.csv").to_string(index=False))

 mean_of_aggregate_mean  mean_of_cross_vm_p95  mean_of_cross_vm_p99  mean_of_cross_vm_max  ratio_p95_to_mean  mean_share_vms_above_80pct  mean_share_vms_above_90pct
               9.414089             37.547024             87.268332             99.295121           3.988386                    1.272091                    0.878364

 top_k_vms  share_of_total_cpu_pct
        10                1.042492
       100                9.596431
       500               30.014360
      1000               43.544253


## Composition versus load

Is aggregate movement caused by which VMs are active, or by the same VMs changing
their usage? Decompose the variance: between-VM level, common time effect, and
idiosyncratic residual.

In [8]:
print(pd.read_csv(ART / "movement_decomposition.csv").to_string(index=False))
print()
print(pd.read_csv(ART / "composition_vs_load.csv").to_string(index=False))

 std_aggregate_change_pp  std_within_vm_change_component_pp  corr_aggregate_with_within_component  mean_abs_aggregate_change_pp  mean_abs_within_component_change_pp                                                                                                                                                                                                                note
                0.158033                           0.158033                                   1.0                      0.121117                             0.121117 the between-VM term sums to zero inside each timestamp when every VM is present, so aggregate movement in this release is 100% within-VM load change; composition effects can only enter through membership changes

 var_between_vm_levels  var_common_time_component  var_idiosyncratic  share_between_vm_pct  share_common_time_pct  share_idiosyncratic_pct  n_vms  n_timestamps
            170.324726                   0.037128          33.419576             83.5

83.6% of the variance is simply *which machine you are looking at*. The
common-time component is 0.018%, so there is almost no "the whole fleet got
busier" signal to forecast. That is the core reason the aggregate is so hard to
beat with anything but a naive forecast.

## Operational takeaway

1. Forecast the count trivially - it does not move in this slice.
2. Forecast the aggregate with a strong baseline and honest uncertainty, not with
   a complex model on 110 points.
3. Tell the capacity team about the tail, because the mean is a poor proxy for
   the machine that matters.